In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot
from sklearn.model_selection import train_test_split

In [3]:
# El dataset no tiene encabezado — hay que definir los nombres
columnas = [
    'age', 'workclass', 'fnlwgt', 'education',
    'education_num', 'marital_status', 'occupation',
    'relationship', 'race', 'sex', 'capital_gain',
    'capital_loss', 'hours_per_week', 'native_country', 'income'
]

df = pd.read_csv('./datasets/adult/adult.data', header=None, names=columnas,
                 sep=', ', engine='python')

print(f"Filas:    {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print(f"\nPrimeras 5 filas:")
print(df.head())
print(f"\nTipos de datos:")
print(df.dtypes)

Filas:    32561
Columnas: 15

Primeras 5 filas:
   age         workclass  fnlwgt  education  education_num  \
0   39         State-gov   77516  Bachelors             13   
1   50  Self-emp-not-inc   83311  Bachelors             13   
2   38           Private  215646    HS-grad              9   
3   53           Private  234721       11th              7   
4   28           Private  338409  Bachelors             13   

       marital_status         occupation   relationship   race     sex  \
0       Never-married       Adm-clerical  Not-in-family  White    Male   
1  Married-civ-spouse    Exec-managerial        Husband  White    Male   
2            Divorced  Handlers-cleaners  Not-in-family  White    Male   
3  Married-civ-spouse  Handlers-cleaners        Husband  Black    Male   
4  Married-civ-spouse     Prof-specialty           Wife  Black  Female   

   capital_gain  capital_loss  hours_per_week native_country income  
0          2174             0              40  United-States  <=

In [4]:
# El dataset tiene espacios al inicio en los strings
# ya los limpiamos con sep=', ' pero verificamos
print("Valores únicos en income:")
print(df['income'].unique())

# Reemplazar '?' por NaN
df = df.replace('?', np.nan)

print(f"\nNulos por columna:")
nulos = df.isnull().sum()
print(nulos[nulos > 0])

Valores únicos en income:
['<=50K' '>50K']

Nulos por columna:
workclass         1836
occupation        1843
native_country     583
dtype: int64


In [5]:
# Las columnas con '?' son categóricas
# imputar con la moda (valor más frecuente)
cols_con_nulos = df.columns[df.isnull().any()].tolist()
print(f"Columnas con nulos: {cols_con_nulos}")

for col in cols_con_nulos:
    moda = df[col].mode()[0]
    df[col] = df[col].fillna(moda)
    print(f"  {col} → imputado con moda: '{moda}'")

print(f"\nNulos restantes: {df.isnull().sum().sum()}")

Columnas con nulos: ['workclass', 'occupation', 'native_country']
  workclass → imputado con moda: 'Private'
  occupation → imputado con moda: 'Prof-specialty'
  native_country → imputado con moda: 'United-States'

Nulos restantes: 0


In [6]:
#Variables numéricas y categóricas
numericas   = df.select_dtypes(include='number').columns.tolist()
categoricas = df.select_dtypes(include='object').columns.tolist()

print(f"Variables numéricas ({len(numericas)}):")
print(numericas)
print(f"\nVariables categóricas ({len(categoricas)}):")
print(categoricas)

# Ver valores únicos de cada categórica
for col in categoricas:
    n = df[col].nunique()
    print(f"\n{col} — {n} valores únicos:")
    print(df[col].value_counts())

Variables numéricas (6):
['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']

Variables categóricas (9):
['workclass', 'education', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'native_country', 'income']

workclass — 8 valores únicos:
workclass
Private             24532
Self-emp-not-inc     2541
Local-gov            2093
State-gov            1298
Self-emp-inc         1116
Federal-gov           960
Without-pay            14
Never-worked            7
Name: count, dtype: int64

education — 16 valores únicos:
education
HS-grad         10501
Some-college     7291
Bachelors        5355
Masters          1723
Assoc-voc        1382
11th             1175
Assoc-acdm       1067
10th              933
7th-8th           646
Prof-school       576
9th               514
12th              433
Doctorate         413
5th-6th           333
1st-4th           168
Preschool          51
Name: count, dtype: int64

marital_status — 7 valores únicos:
marital_statu

In [7]:
# Variable objetivo — income: <=50K o >50K
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})
print("income codificado: <=50K=0, >50K=1")
print(df['income'].value_counts())

# sex — 2 valores, codificación manual
df['sex'] = df['sex'].map({'Male': 0, 'Female': 1})
print("\nsex codificado: Male=0, Female=1")

# education ya tiene education_num que es su versión numérica
# podemos eliminar education y quedarnos con education_num
df = df.drop(columns=['education'])
print("\nColumna 'education' eliminada — reemplazada por 'education_num'")

# fnlwgt es un peso estadístico — no es útil para predecir
df = df.drop(columns=['fnlwgt'])
print("Columna 'fnlwgt' eliminada — peso estadístico sin valor predictivo")

# Las demás categóricas tienen muchos valores — One Hot Encoding
cols_ohe = ['workclass', 'marital_status', 'occupation',
            'relationship', 'race', 'native_country']

df = pd.get_dummies(df, columns=cols_ohe, drop_first=True)
print(f"\nShape después de One Hot Encoding: {df.shape}")

income codificado: <=50K=0, >50K=1
income
0    24720
1     7841
Name: count, dtype: int64

sex codificado: Male=0, Female=1

Columna 'education' eliminada — reemplazada por 'education_num'
Columna 'fnlwgt' eliminada — peso estadístico sin valor predictivo

Shape después de One Hot Encoding: (32561, 82)


In [10]:
#Split del dataset en X e Y
y = df['income'].values
X = df.drop(columns=['income']).values

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# Split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Normalizar con datos de train
def featureNormalize(X):
    mu    = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    sigma[sigma == 0] = 1
    return (X - mu) / sigma, mu, sigma

# Convertir booleanos y enteros a float para que NumPy sea feliz
X_train = X_train.astype(float)
X_test = X_test.astype(float)

X_train_norm, mu, sigma = featureNormalize(X_train)
X_test_norm  = (X_test - mu) / sigma

print(f"\nEntrenamiento: {X_train_norm.shape}")
print(f"Prueba:        {X_test_norm.shape}")
print(f"\nMedia X_train_norm (debe ser ~0): {X_train_norm.mean():.6f}")
print(f"Std X_train_norm  (debe ser ~1): {X_train_norm.std():.6f}")

X shape: (32561, 81)
y shape: (32561,)

Entrenamiento: (26048, 81)
Prueba:        (6513, 81)

Media X_train_norm (debe ser ~0): -0.000000
Std X_train_norm  (debe ser ~1): 1.000000
